In [ ]:
import re
import os
import sys
import json
import regex
import pandas as pd
import logging
from transformers import AutoTokenizer, AutoModel

In [58]:
os.environ["CUDA_VISIBLE_DEVICES"] = ""        # ép chạy CPU
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"       # debug stacktrace đồng bộ (giữ lại cả khi về GPU sau này)

import torch
import torch.nn.functional as F

In [59]:
# print(torch.__version__)
# print(torch.cuda.is_available())

In [60]:
# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)

In [ ]:
# 1. Danh sách từ khóa toxic (regex)
VI_TOXIC_LEXICON = [
    "địt","đụ","đéo","đếch","đcm","đm","vãi","vkl","vcl","vãi l*n","vãi lồn",
    "đồ ngu","ngu","đần","đần độn","óc chó","óc lợn","óc heo","khốn","khốn nạn",
    "cặn bã","rác rưởi","bẩn thỉu","bẩn bựa","cút","cút mẹ","cút đi",
    "dm","cc","clgt","má mày","mẹ mày","cha mày","bố mày","tao","mày",
    "lồn","loz","cặc","đĩ","con đĩ","điếm","gái điếm","thằng chó","thằng",
    "ngu học","thằng đần","bitch","fuck","wtf","asshole","dick","pussy","bastard"
]

# Compile regex (case-insensitive, có xử lý dấu cách)
toxic_patterns = [re.compile(rf"\b{re.escape(word)}\b", re.IGNORECASE) for word in VI_TOXIC_LEXICON]

In [ ]:
# 2. Hàm regex check
def detect_toxic_regex(text: str):
    found = []
    for pat, word in zip(toxic_patterns, VI_TOXIC_LEXICON):
        if pat.search(text):
            found.append(word)
    return found

In [ ]:
# 3. Load PhoBERT v2
# Xác định thiết bị (GPU nếu có)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# logging.info(f"Thiết bị sử dụng: {device}")

# Load mô hình
custom_cache_dir = "D:/Model"
model_name = "vinai/phobert-base-v2"

os.makedirs(custom_cache_dir, exist_ok=True)
logging.info("⏳ Đang load PhoBERT v2...")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=custom_cache_dir,
    use_fast=False,          # ⚠️ rất quan trọng với PhoBERT
    local_files_only=False
)

model = AutoModel.from_pretrained(
    model_name,
    cache_dir=custom_cache_dir,
    local_files_only=False
)

# Chuyển mô hình sang thiết bị
# model = model.to(device)

model.eval()
logging.info("✅ PhoBERT v2 đã sẵn sàng")

2025-10-02 18:46:26,767 - INFO - ⏳ Đang load PhoBERT v2...


Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


2025-10-02 18:46:29,363 - INFO - ✅ PhoBERT v2 đã sẵn sàng


In [64]:
def clean_text(text: str):
    # Chỉ giữ Latin + dấu tiếng Việt + khoảng trắng + dấu câu + số
    text = regex.sub(r"[^\p{Latin}\p{M}\p{Zs}\p{P}\p{N}]", " ", str(text))
    return text.lower().strip()

In [65]:
# 4. Hàm lấy embedding
def get_embedding(text: str, max_len: int = 256):
    # Chuẩn hóa văn bản
    norm_text = clean_text(text)
    if not norm_text.strip():
        logging.warning("Văn bản sau khi làm sạch là rỗng!")
        return torch.zeros(1, model.config.hidden_size)  # Văn bản rỗng → trả vector 0 để không gây lỗi
    
    # Tokenize
    inputs = tokenizer(
        norm_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_len,
    )
    vocab_size = model.get_input_embeddings().weight.size(0)
    if inputs["input_ids"].max().item() >= vocab_size:
        logging.error(f"ID vượt vocab (max={inputs['input_ids'].max().item()} >= {vocab_size}), skip CPU fallback.")
        return torch.zeros((1, model.config.hidden_size))
    
    # Chuyển inputs sang thiết bị phù hợp
    # inputs = {k: v for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)  # CPU
    return outputs.last_hidden_state.mean(dim=1)
    # return emb


In [66]:
# Precompute embeddings cho nhãn
toxic_emb = get_embedding("toxic ngôn ngữ độc hại")
nontoxic_emb = get_embedding("non-toxic ngôn ngữ lành mạnh")

In [67]:
def classify_with_phobert(text: str):
    emb = get_embedding(text)
    sim_toxic = F.cosine_similarity(emb, toxic_emb).item()
    sim_non   = F.cosine_similarity(emb, nontoxic_emb).item()
    return "toxic" if sim_toxic > sim_non else "non-toxic"

In [68]:
# # 4. Hàm detect toxic bằng mô hình
# def detect_toxic_model(text: str, max_new_tokens: int = 64):
#     prompt = f"""
#     Hãy phân loại câu sau có chứa ngôn ngữ độc hại/toxic hay không.
#     Trả lời chỉ với: "TOXIC" hoặc "NON-TOXIC".
    
#     Câu: "{text}"
#     """
#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             temperature=0.0,
#             do_sample=False
#         )
#     result = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     return "TOXIC" if "TOXIC" in result else "NON-TOXIC"

In [ ]:
def check_post_toxic(post_text: str, index: int):
    if not post_text or not isinstance(post_text, str) or post_text.strip() == "":
        logging.warning(f"[{index}] Văn bản rỗng hoặc không hợp lệ, bỏ qua...")
        return {
            "index": index,
            "post_text": post_text,
            "method": "skipped",
            "status": "invalid",
            "toxic_word": None,
        }
    
    regex_found = detect_toxic_regex(post_text)
    if regex_found:
        logging.info(f"[{index}] Regex phát hiện: {regex_found}")
        return {
            "index": index,
            "post_text": post_text,
            "method": "regex",
            "status": "toxic",
            "toxic_word": regex_found,
        }
    else:
        try:
            status = classify_with_phobert(post_text)
            logging.info(f"[{index}] PhoBERT phân loại: {status}")
            return {
                "index": index,
                "post_text": post_text,
                "method": "phobert",
                "status": status,
                "toxic_word": None,
            }
        except Exception as e:
            logging.error(f"[{index}] Lỗi khi phân loại PhoBERT: {str(e)}")
            return {
                "index": index,
                "post_text": post_text,
                "method": "error",
                "status": "failed",
                "toxic_word": None,
            }

In [70]:
if __name__ == "__main__":
    # 1. Nhập đường dẫn file
    input_path = "Confessions of HNMU.xlsx"
    output_path = "output_toxic.json"
    col_name = "post_text"

    # 2. Đọc file Excel
    df = pd.read_excel(input_path).head(2)
    df[col_name] = df[col_name].fillna("")  # Thay NaN bằng chuỗi rỗng

    if col_name not in df.columns:
        raise ValueError(f"Không tìm thấy cột '{col_name}' trong file Excel.")

    # 3. Ghi JSON
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("[\n")
        for idx, text in enumerate(df[col_name].astype(str).tolist()):
            logging.info(f"🔎 Đang xử lý post {idx+1}/{len(df)}; nội dung (rút gọn): {str(text)[:60]}...")
            try:
                result = check_post_toxic(text, idx)
            except Exception as e:
                logging.error(f"[{idx}] Lỗi classify: {e}")
                result = {
                    "index": idx,
                    "post_text": text,
                    "method": "phobert",
                    "status": "error",
                    "toxic_word": None,
                    "error": str(e)
                }

            line = json.dumps(result, ensure_ascii=False, indent=4)
            if idx > 0: f.write(",\n")
            f.write(line); f.flush()
        f.write("\n]\n")

    logging.info(f"✅ Kết quả đã được lưu vào: {output_path}")




2025-10-02 18:46:29,666 - INFO - 🔎 Đang xử lý post 1/2; nội dung (rút gọn): Địt mẹ mày...
2025-10-02 18:46:29,699 - INFO - [0] PhoBERT phân loại: non-toxic
2025-10-02 18:46:29,700 - INFO - 🔎 Đang xử lý post 2/2; nội dung (rút gọn): 8149: Mình được các cô ở phòng CTQLHSSV nhắc là báo với các ...
2025-10-02 18:46:29,817 - INFO - [1] PhoBERT phân loại: non-toxic
2025-10-02 18:46:29,818 - INFO - ✅ Kết quả đã được lưu vào: output_toxic.json
